In [2]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.8 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [4]:
data = pd.read_csv('/content/drive/MyDrive/vkr/annotations.csv', sep='\t', on_bad_lines='skip')
data

,attachment_id,text,user_id,height,width,length,train,begin,end
0,44e8d2a0-7e01-450b-90b0-beb7400d2c1e,Ё,185bd3a81d9d618518d10abebf0d17a8,1920,1080,156.0,True,36,112
1,df5b08f0-41d1-4572-889c-8b893e71069b,А,185bd3a81d9d618518d10abebf0d17a8,1920,1080,150.0,True,36,76
2,17f53df4-c467-4aff-9f48-20687b63d49a,Р,185bd3a81d9d618518d10abebf0d17a8,1920,1080,133.0,True,40,97
3,e3add916-c708-4339-ad98-7e2740be29e9,Е,185bd3a81d9d618518d10abebf0d17a8,1920,1080,144.0,True,43,107
4,bd7272ed-1850-48f1-a2a8-c8fed523dc37,Ч,185bd3a81d9d618518d10abebf0d17a8,1920,1080,96.0,True,20,70
...,...,...,...,...,...,...,...,...,...
20395,nodca88242-2bc7-4a77-9d14-103aa1dacbd6,no_event,0041ec866777f12c384b64d8cd636277,1920,1080,42.0,False,0,42
20396,no7a7812b1-ae64-4402-9ebc-5b947edbd021,no_event,f5b82a9c82f6d870ec253e4c3fa96d83,1280,720,32.0,False,0,32
20397,no62a4df76-b48d-4f61-b6e8-bc4a1eb0cb61,no_event,4299b8ccf39ace57287b463fbe4a489b,1920,960,32.0,False,0,32
20398,no388a3f7c-3594-4332-bc78-b5b53190301d,no_event,c80b4e57f158f28299b2a89694c42329,1920,1080,41.0,False,0,41


In [5]:
source_directory = '/content' # базовая папка
train_dir = os.path.join(source_directory, 'train')
test_dir = os.path.join(source_directory, 'test')

In [6]:
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

In [7]:
all_class_counts = data['text'].value_counts()
top_classes = all_class_counts.index.tolist()
data = data[data['text'].isin(top_classes)]

# Словарь меток только для этих классов
class_names = sorted(top_classes)
label2idx = {label: idx for idx, label in enumerate(class_names)}
idx2label = {idx: label for idx, label in enumerate(class_names)}

In [8]:
# Параметры

SEQUENCE_LENGTH = 48
MAX_HANDS = 2
NUM_LANDMARKS = 21 #количество точек для каждой руки
NUM_FEATURES_PER_LANDMARK_HAND = 3 #количество координат для каждой точки
NUM_FEATURES = MAX_HANDS * NUM_LANDMARKS * NUM_FEATURES_PER_LANDMARK_HAND  #21 точка по 3 координаты

NUM_FEATURES_PER_LANDMARK = 2
LIPS_IDX = [61, 37, 0, 267, 291, 405, 17, 181] #точки губ
NUM_FEATURES_FACE = len(LIPS_IDX)

POSE_IDX = [16, 14, 12, 11, 13, 15, 0] #точки рук + нос
NUM_FEATURES_POSE = len(POSE_IDX)

NUM_FEATURES_ALL = NUM_FEATURES + NUM_FEATURES_FACE * NUM_FEATURES_PER_LANDMARK + NUM_FEATURES_POSE * NUM_FEATURES_PER_LANDMARK


In [11]:
def find_npy_path(attachment_id, is_train=True):
  folder = train_dir if is_train else test_dir
  path = os.path.join(folder, f"{attachment_id}.npy")
  if os.path.exists(path):
    return path
  return None

In [13]:
class GestureDataset(Dataset):
    def __init__(self, data, transform=None):
        """
        data_dir: путь к папке, где лежат подпапки с классами
        class_names: список названий классов, например ['no-gesture', 'да', 'нет', ...]
        """
        self.data = []
        self.labels = []
        self.transform = transform

        for idx, row in tqdm(data.iterrows(), total=len(data)):
            attachment_id = row['attachment_id']

            video_path = find_npy_path(attachment_id, is_train=True)
            if video_path:
                seq = np.load(video_path)
                self.data.append(seq)
                self.labels.append(label2idx[row['text']])

        print(f"Загружено {len(self.data)} примеров")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]
        label = self.labels[idx]

        if self.transform:
            seq = self.transform(seq)

        return torch.from_numpy(seq).float(), torch.tensor(label, dtype=torch.long)



In [14]:
def load_preprocessed_gesture_dataset(save_path='/content/drive/MyDrive/vkr/gesture_dataset_preprocessed.pth',
                                      batch_size=64):

    saved = torch.load(save_path, weights_only=False)

    # Создаём датасет с уже готовыми данными
    dataset = GestureDataset.__new__(GestureDataset)   # создаём объект без вызова __init__
    dataset.data = saved['data']
    dataset.labels = saved['labels']
    dataset.transform = None

    print(f"Загружено {len(dataset)} примеров из файла")

    # Subset'ы
    train_dataset = torch.utils.data.Subset(dataset, saved['train_idx'])
    val_dataset   = torch.utils.data.Subset(dataset, saved['val_idx'])

    # DataLoader'ы
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
        drop_last=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )

    return train_loader, val_loader, dataset

In [16]:
train_loader, val_loader, full_dataset = load_preprocessed_gesture_dataset()

Загружено 15300 примеров из файла


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [25]:
@torch.no_grad()
def validate(model, val_loader, device, criterion):
    model.eval()

    total_loss = 0.0
    top1_correct = 0
    top5_correct = 0
    total_samples = 0

    for sequences, labels in val_loader:
        sequences = sequences.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(sequences)          # [B, num_classes]

        loss = criterion(outputs, labels)
        total_loss += loss.item() * labels.size(0)

        # Топ-1 и Топ-5
        _, pred_top5 = outputs.topk(5, dim=1, largest=True, sorted=True)

        top1_correct += (pred_top5[:, 0] == labels).sum().item()
        top5_correct += (pred_top5 == labels.view(-1, 1)).any(dim=1).sum().item()

        total_samples += labels.size(0)

    # Итоговые метрики
    avg_loss = total_loss / total_samples
    top1_acc = (top1_correct / total_samples) * 100
    top5_acc = (top5_correct / total_samples) * 100

    print(f"Validation Results:")
    print(f"   Loss: {avg_loss:.4f}")
    print(f"   Top-1 Accuracy: {top1_acc:.2f}%")
    print(f"   Top-5 Accuracy: {top5_acc:.2f}%")

    return top1_acc, top5_acc, avg_loss

In [22]:
def mixup_data(x, y, alpha=0.2):
    """Mixup аугментация"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Loss для mixup"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LandmarkEmbedding(nn.Module):
    def __init__(self, in_features, d_model=256, dropout=0.2):
        super().__init__()
        self.proj = nn.Linear(in_features, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: [B, T, F]
        x = self.proj(x)
        x = self.norm(x)

        attn_out, _ = self.attn(x, x, x)
        x = x + self.dropout(attn_out)
        x = self.norm2(x + self.ffn(x))

        x = x.mean(dim=1)   # [B, d_model]
        return x


class GestureModel(nn.Module):
    def __init__(self, num_classes=1001, d_model=256, dropout=0.25):
        super().__init__()

        self.d_model = d_model

        self.hand_emb = LandmarkEmbedding(in_features=126, d_model=d_model, dropout=dropout)
        self.lips_emb  = LandmarkEmbedding(in_features=16,  d_model=d_model, dropout=dropout)
        self.pose_emb  = LandmarkEmbedding(in_features=14,  d_model=d_model, dropout=dropout)

        self.fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model * 2),
            nn.LayerNorm(d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

        self.temporal_pos = nn.Parameter(torch.randn(1, 300, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=8,
            dim_feedforward=d_model*3,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)

        self.pooling = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        B, T, F = x.shape

        hands = x[:, :, :126]      
        lips  = x[:, :, 126:142]  
        pose  = x[:, :, 142:]

        h = self.hand_emb(hands)
        l = self.lips_emb(lips)
        p = self.pose_emb(pose)

        fused = torch.cat([h, l, p], dim=1) 
        fused = self.fusion(fused)           

        x = fused.unsqueeze(1) + self.temporal_pos[:, :1, :]

        x = self.transformer(x)
        x = x.squeeze(1)                     

        logits = self.classifier(x)

        return logits

In [ ]:
num_epochs = 300

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GestureModel(num_classes=1001, d_model=384, dropout=0.3).to(device) 

criterion = nn.CrossEntropyLoss(label_smoothing=0.2)   
optimizer = optim.AdamW(model.parameters(),
                       lr=8e-4,
                       weight_decay=5e-2)          

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=2.5e-3,
    epochs=num_epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.15,
    div_factor=10,
    final_div_factor=1000
)

best_val_top1 = 0.0
mixup_alpha = 0.3                     

for epoch in tqdm(range(num_epochs)):
    model.train()
    train_loss = 0.0

    for sequences, labels in train_loader:
        sequences = sequences.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        if epoch >= 5 and torch.rand(1).item() < 0.8:     
            mixed_seq, y_a, y_b, lam = mixup_data(sequences, labels, alpha=mixup_alpha)
            outputs = model(mixed_seq)
            loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
        else:
            outputs = model(sequences)
            loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    if (epoch + 1) % 2 == 0 or epoch == num_epochs - 1:
        val_top1, val_top5, val_loss = validate(model, val_loader, device, criterion)

        print(f"Epoch [{epoch+1:3d}/{num_epochs}] "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Top-1: {val_top1:.2f}% | Top-5: {val_top5:.2f}% | "
              f"LR: {scheduler.get_last_lr()[0]:.2e}")

        if val_top1 > best_val_top1:
            best_val_top1 = val_top1
            torch.save(model.state_dict(),
                      f"/content/drive/MyDrive/vkr/epochs/300_best_model_epoch_{epoch+1:03d}_top1_{val_top1:.2f}.pth")
            print(f"    → Новая лучшая модель! Top-1 = {val_top1:.2f}%")
    else:
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] Train Loss: {avg_train_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

/tmp/ipykernel_10384/2157078106.py:75: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
  0%|          | 1/300 [00:08<40:39,  8.16s/it]

Epoch [  1/300] Train Loss: 6.8964 | LR: 2.53e-04


  1%|          | 2/300 [00:16<39:56,  8.04s/it]

Validation Results:
   Loss: 6.8287
   Top-1 Accuracy: 2.22%
   Top-5 Accuracy: 2.40%
Epoch [  2/300] Train Loss: 6.8345 | Val Loss: 6.8287 | Top-1: 2.22% | Top-5: 2.40% | LR: 2.61e-04
    → Новая лучшая модель! Top-1 = 2.22%


  1%|          | 3/300 [00:23<39:12,  7.92s/it]

...

Validation Results:
   Loss: 4.2282
   Top-1 Accuracy: 32.98%
   Top-5 Accuracy: 64.40%
Epoch [298/300] Train Loss: 3.4358 | Val Loss: 4.2282 | Top-1: 32.98% | Top-5: 64.40% | LR: 6.28e-07


100%|█████████▉| 299/300 [40:12<00:07,  7.99s/it]

Epoch [299/300] Train Loss: 3.2958 | LR: 3.44e-07


100%|██████████| 300/300 [40:20<00:00,  8.07s/it]

Validation Results:
   Loss: 4.2283
   Top-1 Accuracy: 33.03%
   Top-5 Accuracy: 64.36%
Epoch [300/300] Train Loss: 3.5258 | Val Loss: 4.2283 | Top-1: 33.03% | Top-5: 64.36% | LR: 2.50e-07
